# Fraud Detection — Data Preprocessing

Preprocessing pipeline for `transactions.csv`, tailored for tree-based models:
- **XGBoost**, **Random Forest**, **Decision Trees**

### Preprocessing Strategy
| Feature Type | Method |
|---|---|
| Skewed numeric (`amt`, `city_pop`) | Log-transform |
| Ordinal categorical (`income_band`, `education`, `merchant_size`) | Ordinal integer encoding |
| Nominal categorical (`mcc`, `state`, `job`, `gender`, `marital_status`) | One-hot encoding |
| Date (`dob`) | Convert to age (years) |
| Identifier (`cc_num`, `city`) | Drop |
| Other numeric | Passthrough |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer, OrdinalEncoder, OneHotEncoder
)

SEED = 42
TEST_SIZE = 0.2

## 1. Load Data & Select Features

In [ ]:
DATA_PATH = "Sparkov_Data_Generation/output_data/transactions.csv"

# All columns we need from the CSV
USE_COLS = [
    "unix_time", "transaction_velocity", "amt", "mcc",
    "lat", "long", "merch_lat", "merch_long",
    "cc_num", "dob", "gender", "job",
    "income_band", "education", "marital_status",
    "city", "state", "city_pop", "merchant_size",
    "is_fraud",
]

df = pd.read_csv(DATA_PATH, sep="|", usecols=USE_COLS)
print(f"{len(df):,} rows | {df['is_fraud'].sum():,} fraud ({df['is_fraud'].mean():.3%} base rate)")
df.head()

## 2. Feature Engineering

Before the pipeline, we handle transformations that create or remove columns.

In [ ]:
# --- Convert dob -> age (in years, relative to the transaction time) ---
df["dob"] = pd.to_datetime(df["dob"])
# Use the median transaction date as reference point
reference_date = pd.to_datetime(df["unix_time"].median(), unit="s")
df["age"] = (reference_date - df["dob"]).dt.days / 365.25
df["age"] = df["age"].round(1)

print(f"Age range: {df['age'].min():.1f} – {df['age'].max():.1f} years")
print(f"Age mean:  {df['age'].mean():.1f}, std: {df['age'].std():.1f}")

# --- Drop columns we don't need ---
# cc_num: identifier, not a feature
# city: too many unique values (438), risk of overfitting
# dob: replaced by age
df = df.drop(columns=["cc_num", "city", "dob"])

# --- Cast mcc to string (it's a category code, not a number) ---
df["mcc"] = df["mcc"].astype(str)

print(f"\nFinal columns ({len(df.columns)}): {list(df.columns)}")
df.head()

## 3. Train/Test Split

Stratified split to preserve the fraud rate in both sets.

In [ ]:
TARGET = "is_fraud"
y = df[TARGET]
X = df.drop(columns=[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print(f"Train: {len(X_train):,} rows ({y_train.mean():.3%} fraud)")
print(f"Test:  {len(X_test):,} rows  ({y_test.mean():.3%} fraud)")

## 4. Preprocessing Pipeline

### Strategy by feature group:

| Group | Features | Transform | Rationale |
|-------|----------|-----------|----------|
| **Log-transform** | `amt`, `city_pop` | `log1p` | Heavily right-skewed (skewness > 1.4), log spreads out clustered values for better tree splits |
| **Passthrough** | `unix_time`, `transaction_velocity`, `lat`, `long`, `merch_lat`, `merch_long`, `age` | None | Moderate variance/skewness, trees handle fine as-is |
| **Ordinal encode** | `income_band`, `education`, `merchant_size` | Integer mapping (0, 1, 2) | Ranked features where order matters |
| **One-hot encode** | `mcc`, `state`, `job`, `gender`, `marital_status` | Binary columns | Nominal categories with no meaningful order |

In [ ]:
# ============================================================
# Define column groups
# ============================================================

# Skewed numeric -> log-transform
LOG_COLS = ["amt", "city_pop"]

# Numeric -> passthrough (no scaling needed for tree models)
PASSTHROUGH_COLS = [
    "unix_time", "transaction_velocity",
    "lat", "long", "merch_lat", "merch_long",
    "age",
]

# Ordinal categoricals -> integer encoding preserving rank
ORDINAL_COLS = ["income_band", "education", "merchant_size"]
ORDINAL_CATEGORIES = [
    ["<50k", "50-100k", "100k+"],                       # income_band: low -> high
    ["hs_or_less", "some_college", "bachelor_plus"],     # education: low -> high
    ["small", "medium", "large"],                        # merchant_size: small -> large
]

# Nominal categoricals -> one-hot encoding
ONEHOT_COLS = ["mcc", "state", "job", "gender", "marital_status"]

print(f"Log-transform:    {len(LOG_COLS)} cols  -> {LOG_COLS}")
print(f"Passthrough:      {len(PASSTHROUGH_COLS)} cols  -> {PASSTHROUGH_COLS}")
print(f"Ordinal encode:   {len(ORDINAL_COLS)} cols  -> {ORDINAL_COLS}")
print(f"One-hot encode:   {len(ONEHOT_COLS)} cols  -> {ONEHOT_COLS}")

In [ ]:
# ============================================================
# Build the ColumnTransformer
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        # 1. Log-transform skewed numeric features
        #    Using log1p (log(1+x)) to handle zeros safely
        ("log", FunctionTransformer(np.log1p, validate=True), LOG_COLS),

        # 2. Passthrough numeric features (no transformation)
        ("num", "passthrough", PASSTHROUGH_COLS),

        # 3. Ordinal encoding for ranked categorical features
        ("ord", OrdinalEncoder(categories=ORDINAL_CATEGORIES,
                               handle_unknown="use_encoded_value",
                               unknown_value=-1),
         ORDINAL_COLS),

        # 4. One-hot encoding for nominal categorical features
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         ONEHOT_COLS),
    ],
    remainder="drop",  # Drop any column not explicitly listed
    verbose_feature_names_out=True,
)

print("Pipeline configured. Fitting on training data...")

In [ ]:
# ============================================================
# Fit on training data, transform both sets
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Get feature names for interpretability
feature_names = preprocessor.get_feature_names_out()

# Convert to DataFrames for inspection
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print(f"Processed feature count: {X_train_df.shape[1]}")
print(f"Train shape: {X_train_df.shape}")
print(f"Test shape:  {X_test_df.shape}")
print(f"\nFeature names ({len(feature_names)}):\n{list(feature_names)}")

## 5. Verify Preprocessing

In [ ]:
# ============================================================
# Verify log-transform reduced skewness
# ============================================================

print("=== Skewness Comparison (before vs after log-transform) ===")
for col in LOG_COLS:
    before = X_train[col].skew()
    log_col_name = f"log__{col}"
    after = X_train_df[log_col_name].skew()
    print(f"  {col:15s}  before: {before:+.3f}  ->  after: {after:+.3f}")

print("\n=== Ordinal Encoding Verification ===")
for col, cats in zip(ORDINAL_COLS, ORDINAL_CATEGORIES):
    ord_col = f"ord__{col}"
    print(f"  {col}: {cats} -> mapped to {sorted(X_train_df[ord_col].unique())}")

print(f"\n=== One-Hot Columns Created ===")
ohe_cols = [c for c in feature_names if c.startswith("ohe__")]
print(f"  {len(ohe_cols)} one-hot columns")
print(f"  Examples: {ohe_cols[:6]}...")

In [ ]:
# ============================================================
# Summary statistics of processed features
# ============================================================

# Show stats for the non-OHE columns (more interpretable)
non_ohe_cols = [c for c in feature_names if not c.startswith("ohe__")]
print("=== Processed Feature Statistics (non-OHE columns) ===")
X_train_df[non_ohe_cols].describe().round(3)

In [ ]:
# ============================================================
# Check for nulls/NaNs
# ============================================================

null_counts = X_train_df.isnull().sum()
if null_counts.sum() == 0:
    print("No missing values in processed features")
else:
    print("Missing values found:")
    print(null_counts[null_counts > 0])

## 6. Export Preprocessed Data

Save processed data and the fitted preprocessor for use in modeling.

In [ ]:
import joblib
import os

OUTPUT_DIR = "preprocessed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save processed DataFrames
X_train_df.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False)
X_test_df.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)

# Save the fitted preprocessor (for applying to new data later)
joblib.dump(preprocessor, f"{OUTPUT_DIR}/preprocessor.joblib")

print(f"Saved to {OUTPUT_DIR}/")
print(f"   X_train.csv  ({X_train_df.shape})")
print(f"   X_test.csv   ({X_test_df.shape})")
print(f"   y_train.csv  ({len(y_train):,} rows)")
print(f"   y_test.csv   ({len(y_test):,} rows)")
print(f"   preprocessor.joblib")

In [ ]:
# ============================================================
# Quick peek at final processed data
# ============================================================

print(f"Final processed training data: {X_train_df.shape[0]:,} rows x {X_train_df.shape[1]} features")
print(f"\nFirst 5 rows (all columns):")
X_train_df.head()